# Verdia prep 01 — download PrimeKG & build the compact graph

**What this does:** downloads Harvard's PrimeKG knowledge graph (~4M biomedical
relationships), filters it to the node types we reason over, and saves the compact
artifacts the Verdia app needs locally (`edges`, node name→id indexes).

**Runtime:** any runtime works (CPU is fine — no training here). ~10–15 min,
mostly the download.

**At the end:** the last cell zips `artifacts_prep/` — download the zip and drop the
files into the repo's `artifacts/` folder (or copy from Drive if mounted).

In [ ]:
# Config: node types we keep (both endpoints must be in this set).
# Everything else (exposures, etc.) is dropped to keep the graph compact.
KEEP_TYPES = {
    'drug', 'disease', 'gene/protein', 'molecular_function',
    'biological_process', 'cellular_component', 'pathway', 'anatomy', 'effect/phenotype',
}
OUT_DIR = 'artifacts_prep'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('kept node types:', sorted(KEEP_TYPES))

In [ ]:
# Download PrimeKG kg.csv from Harvard Dataverse (official access API — allowed use).
import os
if not os.path.exists('kg.csv'):
    !wget -q --show-progress -O kg.csv 'https://dataverse.harvard.edu/api/access/datafile/6180620'
size_mb = os.path.getsize('kg.csv') / 1e6
print(f'kg.csv downloaded: {size_mb:.0f} MB')
assert size_mb > 50, 'Download looks too small — check the Dataverse URL.'

In [ ]:
# Load and INSPECT before trusting anything.
import pandas as pd
kg = pd.read_csv('kg.csv', low_memory=False)
print('shape:', kg.shape)
print('columns:', list(kg.columns))
display(kg.head(3))
for col in ['x_type', 'y_type']:
    if col in kg.columns:
        print(f'\n--- {col} counts ---')
        print(kg[col].value_counts())
if 'relation' in kg.columns:
    print('\n--- top relations ---')
    print(kg['relation'].value_counts().head(40))

In [ ]:
# Filter to kept node types; normalize to a canonical edge list.
# Works even if column names differ slightly — we map defensively.
cols = {c.lower(): c for c in kg.columns}
def pick(*cands):
    for c in cands:
        if c in cols: return cols[c]
    raise KeyError(f'none of {cands} in {list(kg.columns)}')

X_TYPE, X_ID, X_NAME = pick('x_type','x_index_type'), pick('x_id','x_index_id'), pick('x_name','x_index_name')
Y_TYPE, Y_ID, Y_NAME = pick('y_type','y_index_type'), pick('y_id','y_index_id'), pick('y_name','y_index_name')
REL, DREL = pick('relation'), pick('display_relation')

edges = kg[
    kg[X_TYPE].isin(KEEP_TYPES) & kg[Y_TYPE].isin(KEEP_TYPES)
][[X_TYPE, X_ID, X_NAME, REL, DREL, Y_TYPE, Y_ID, Y_NAME]].copy()
edges.columns = ['src_type','src_id','src_name','relation','display_relation','dst_type','dst_id','dst_name']
edges = edges.drop_duplicates()
print(f'kept {len(edges):,} of {len(kg):,} edges')
print('node-type pairs kept (top 15):')
print(edges.groupby(['src_type','dst_type']).size().sort_values(ascending=False).head(15))

In [ ]:
# Node indexes: name -> (type, id) dictionaries, saved per type.
import itertools
frames = []
for side, (tcol, icol, ncol) in [('src', ('src_type','src_id','src_name')), ('dst', ('dst_type','dst_id','dst_name'))]:
    f = edges[[tcol, icol, ncol]].copy()
    f.columns = ['node_type', 'node_id', 'node_name']
    frames.append(f)
nodes = pd.concat(frames).drop_duplicates()
nodes = nodes.sort_values('node_name')
print(f'{len(nodes):,} unique nodes')
print(nodes['node_type'].value_counts())
nodes.to_csv(f'{OUT_DIR}/nodes.csv', index=False)
# per-type name lists for the resolver (exact + fuzzy match later)
for t, grp in nodes.groupby('node_type'):
    safe = t.replace('/', '_')
    grp[['node_id','node_name']].drop_duplicates().to_csv(f'{OUT_DIR}/nodes_{safe}.csv', index=False)
    print(f'{t}: {len(grp):,} nodes -> nodes_{safe}.csv')

In [ ]:
# SANITY: can we find real entities? (substring search helper)
def find_nodes(q, node_type=None, limit=8):
    m = nodes['node_name'].str.lower().str.contains(q.lower(), na=False)
    if node_type: m &= nodes['node_type'] == node_type
    return nodes[m][['node_type','node_id','node_name']].drop_duplicates().head(limit)

display(find_nodes('metformin', 'drug'))
display(find_nodes('alzheimer', 'disease'))
display(find_nodes('dupilumab', 'drug'))  # newer drug — may be absent; that's OK and informative

In [ ]:
# Save artifacts + zip for transfer to the repo's artifacts/ folder.
edges.to_csv(f'{OUT_DIR}/edges.csv.gz', index=False, compression='gzip')
try:
    nodes.to_parquet(f'{OUT_DIR}/nodes.parquet', index=False)
    edges.to_parquet(f'{OUT_DIR}/edges.parquet', index=False)
except Exception as e:
    print('parquet skipped:', e)
!du -sh {OUT_DIR}
!zip -qr verdia_artifacts.zip {OUT_DIR}
from google.colab import files
print('\n>>> Download starting (verdia_artifacts.zip). Unzip into the repo artifacts/ folder.')
files.download('verdia_artifacts.zip')

## After this notebook

1. Unzip `verdia_artifacts.zip` and copy its contents into the repo's `artifacts/` folder
   (commit if total < ~95MB, which the filtered set usually is).
2. Tell the agent (me) — next steps are the resolver + route engine locally, and
   notebook 02 (PyKEEN embeddings — note: PyKEEN ships a native PrimeKG loader,
   so training can run straight from `kg.csv` too).

Ground rules honored: Harvard Dataverse official API download (no scraping); PrimeKG
cite: Chandak, Huang & Zitnik, *Scientific Data* 2023.